# 06 — Kenya Smallholder Fields, Sentinel-2 with FTW Engine

Delineates smallholder agricultural fields in Central Kenya using Sentinel-2 imagery and the **FTW** engine. Demonstrates `min_area` tuning for smallholder agriculture by comparing results at 100, 500, 1000, and 2500 m² thresholds.

**Estimated runtime:** ~10–20 minutes (1 year, small AOI, GPU).

**Prerequisites:**
```bash
pip install agribound[gee,ftw]
agribound auth --project YOUR_GEE_PROJECT
```

## Configuration

In [ ]:
import json
from pathlib import Path

import agribound

GEE_PROJECT = "YOUR_GEE_PROJECT"  # <-- Replace with your GEE project ID

OUTPUT_DIR = Path("../../outputs/kenya_smallholder")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE = "sentinel2"
ENGINE = "ftw"
YEAR = 2023

## Create Study Area

AOI near Nyeri, Central Kenya — a region dominated by smallholder agriculture.

In [ ]:
aoi = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {
                "type": "Polygon",
                "coordinates": [
                    [
                        [36.90, -0.50],
                        [37.05, -0.50],
                        [37.05, -0.35],
                        [36.90, -0.35],
                        [36.90, -0.50],
                    ]
                ],
            },
            "properties": {"name": "Central Kenya AOI"},
        }
    ],
}
study_area = str(OUTPUT_DIR / "kenya_aoi.geojson")
with open(study_area, "w") as f:
    json.dump(aoi, f)

print(f"Study area: {study_area}")

## Compare `min_area` Thresholds

In [ ]:
thresholds = [100, 500, 1000, 2500]
results = {}

for min_area in thresholds:
    print(f"Delineating with min_area={min_area} m²...", end=" ")
    output_path = OUTPUT_DIR / f"fields_minarea_{min_area}.gpkg"

    gdf = agribound.delineate(
        study_area=study_area,
        source=SOURCE,
        year=YEAR,
        engine=ENGINE,
        output_path=str(output_path),
        gee_project=GEE_PROJECT,
        composite_method="median",
        min_area=min_area,
        simplify=1.0,
    )
    results[min_area] = gdf
    print(f"{len(gdf)} fields")

## Summary

In [ ]:
print(f"{'Threshold (m²)':<16} {'Fields':>6} {'Avg Area (m²)':>14}")
print(f"{'-' * 16} {'-' * 6} {'-' * 14}")
for threshold, gdf in results.items():
    avg_area = gdf["metrics:area"].mean() if "metrics:area" in gdf.columns else 0
    print(f"{threshold:<16} {len(gdf):>6} {avg_area:>14,.0f}")

## Visualization

In [ ]:
from agribound.visualize import show_comparison

m = show_comparison(
    list(results.values()),
    labels=[f"min={t} m²" for t in thresholds],
    basemap="Google.Satellite",
    output_html=str(OUTPUT_DIR / "map_kenya_thresholds.html"),
)
m